In [34]:
import torch
torch.set_printoptions(sci_mode=False)

In [2]:
inputs = torch.randn(6, 3)  # 6 samples, each vector length 3
print(inputs)

tensor([[-0.2728,  0.8483, -0.6835],
        [-0.5035, -0.1803, -1.4679],
        [ 0.1080,  1.9712, -1.5753],
        [-0.5000,  0.6529, -0.7750],
        [-0.2441,  0.4189,  1.3836],
        [-0.3643, -1.2531, -0.5631]])


In [3]:
input_query = inputs[1]
print(input_query)

tensor([-0.5035, -0.1803, -1.4679])


In [5]:
input_1 = inputs[0]

In [6]:
input_query@input_1

tensor(0.9877)

In [17]:
attn_weights = []
for idx, tensor in enumerate(inputs):
    attn_weights.append((inputs[1]@tensor).item())

attn_weights

[tensor(0.9877),
 tensor(2.4409),
 tensor(1.9028),
 tensor(1.2717),
 tensor(-1.9837),
 tensor(1.2359)]

In [26]:
# Normalization using softmax
def softmax_naive(x):
    return torch.exp(x)/ (torch.exp(x).sum(dim=0))

attn_weights = softmax_naive(torch.tensor(attn_weights))
print(attn_weights)

tensor([0.0958, 0.4098, 0.2393, 0.1273, 0.0049, 0.1228])


In [20]:
torch.softmax(torch.tensor(attn_weights), dim=0)

tensor([0.0958, 0.4098, 0.2393, 0.1273, 0.0049, 0.1228])

In [30]:
context_vector = torch.zeros(inputs[1].shape)
for idx, x_i in enumerate(inputs):
    context_vector = context_vector + attn_weights[idx]*x_i

print(context_vector)


tensor([-0.3162,  0.4104, -1.2051])


 A simple self attention mechanism without trainable weights

In [ ]:
attn_scores = torch.empty(6,6)
for i, x_i in enumerate(inputs):
     for j, x_j in enumerate(inputs):
        attn_scores[i, j] = x_i@x_j
print(attn_scores)
# These are attn_scores, not yet normalized

tensor([[ 1.2611,  0.9877,  2.7194,  1.2199, -0.5237, -0.5787],
        [ 0.9877,  2.4409,  1.9028,  1.2717, -1.9837,  1.2359],
        [ 2.7194,  1.9028,  6.3790,  2.4538, -1.3802, -1.6224],
        [ 1.2199,  1.2717,  2.4538,  1.2768, -0.6768, -0.1996],
        [-0.5237, -1.9837, -1.3802, -0.6768,  2.1494, -1.2151],
        [-0.5787,  1.2359, -1.6224, -0.1996, -1.2151,  2.0200]])


In [36]:
attn_scores_ = inputs@inputs.T

In [40]:
# Normalization
attn_weights = torch.softmax(attn_scores, dim=1)
attn_weights_ = torch.softmax(attn_scores_, dim=1)
print(attn_weights)
print(attn_weights_)

tensor([[0.1361, 0.1036, 0.5852, 0.1306, 0.0228, 0.0216],
        [0.0958, 0.4098, 0.2393, 0.1273, 0.0049, 0.1228],
        [0.0243, 0.0108, 0.9455, 0.0187, 0.0004, 0.0003],
        [0.1441, 0.1518, 0.4950, 0.1526, 0.0216, 0.0349],
        [0.0571, 0.0133, 0.0243, 0.0490, 0.8277, 0.0286],
        [0.0436, 0.2677, 0.0154, 0.0637, 0.0231, 0.5865]])
tensor([[0.1361, 0.1036, 0.5852, 0.1306, 0.0228, 0.0216],
        [0.0958, 0.4098, 0.2393, 0.1273, 0.0049, 0.1228],
        [0.0243, 0.0108, 0.9455, 0.0187, 0.0004, 0.0003],
        [0.1441, 0.1518, 0.4950, 0.1526, 0.0216, 0.0349],
        [0.0571, 0.0133, 0.0243, 0.0490, 0.8277, 0.0286],
        [0.0436, 0.2677, 0.0154, 0.0637, 0.0231, 0.5865]])


In [41]:
attn_weights.shape

torch.Size([6, 6])

In [45]:
context_vec = attn_weights@inputs
print(attn_weights.shape)
print(inputs.shape)
print(context_vec)

print(attn_weights[0]@inputs)

torch.Size([6, 6])
torch.Size([6, 3])
tensor([[-0.1048,  1.3181, -1.2488],
        [-0.3162,  0.4104, -1.2051],
        [ 0.0805,  1.8945, -1.5360],
        [-0.1565,  1.1357, -1.2091],
        [-0.2566,  0.4368,  0.9943],
        [-0.3962, -0.6646, -0.7947]])
tensor([-0.1048,  1.3181, -1.2488])


In [ ]:
# Implementing the attention weights with trainable weights
inputs
x_2 = inputs[1]
d_in = inputs.shape[1]
d_out = 2

In [52]:
torch.manual_seed(123)
W_query = torch.nn.Parameter(torch.rand(d_in, d_out))
W_key = torch.nn.Parameter(torch.rand(d_in, d_out))
W_value = torch.nn.Parameter(torch.rand(d_in, d_out))

In [53]:
W_query, W_key, W_value

(Parameter containing:
 tensor([[0.2961, 0.5166],
         [0.2517, 0.6886],
         [0.0740, 0.8665]], requires_grad=True),
 Parameter containing:
 tensor([[0.1366, 0.1025],
         [0.1841, 0.7264],
         [0.3153, 0.6871]], requires_grad=True),
 Parameter containing:
 tensor([[0.0756, 0.1966],
         [0.3164, 0.4017],
         [0.1186, 0.8274]], requires_grad=True))

In [56]:
query_2 = x_2 @ W_query
x_2, query_2

(tensor([-0.5035, -0.1803, -1.4679]),
 tensor([-0.3030, -1.6562], grad_fn=<SqueezeBackward4>))

In [60]:
keys = inputs@W_key
values = inputs@W_value

print(keys.shape, values.shape)

torch.Size([6, 2]) torch.Size([6, 2])


In [59]:
keys, values

(tensor([[-0.0966,  0.1187],
         [-0.5647, -1.1912],
         [-0.1191,  0.3606],
         [-0.1924, -0.1095],
         [ 0.4800,  1.2300],
         [-0.4579, -1.3345]], grad_fn=<MmBackward0>),
 tensor([[ 0.1667, -0.2784],
         [-0.2692, -1.3860],
         [ 0.4451, -0.4903],
         [ 0.0769, -0.4773],
         [ 0.2781,  1.2651],
         [-0.4908, -1.0409]], grad_fn=<MmBackward0>))

In [63]:
attn_scores_2 = query_2 @ keys.T   # query_2(1x3) @ keys.T(3x6)
attn_scores_2

tensor([-0.1673,  2.1440, -0.5612,  0.2396, -2.1826,  2.3490],
       grad_fn=<SqueezeBackward4>)

In [69]:
d_k = keys.shape[1]  
attn_weights_2 = torch.softmax(attn_scores_2/d_k**0.5, dim=0)
print(attn_weights_2, attn_weights_2.shape)


tensor([0.0695, 0.3564, 0.0526, 0.0927, 0.0167, 0.4120],
       grad_fn=<SoftmaxBackward0>) torch.Size([6])


In [70]:
context_vec_2 = attn_weights_2 @ values # attn_weights_2(6) @ values(6,2)
context_vec_2

tensor([-0.2514, -0.9911], grad_fn=<SqueezeBackward4>)

In [96]:
# Implementing a genrelized class
import torch.nn as nn
class SelfAttention(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.d_in = d_in
        self.d_out = d_out
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key   = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))
        self.div = self.d_out**0.5

    def forward(self, x):   # x(6,3) all for example only
        query = x @ self.W_query  #query(6,2)  
        key = x @ self.W_key      #key(6,2)
        value = x @ self.W_value  #value(6,2)

        # Need to calculate the attention scores
        attn_scores = query @ key.T   # (6,2) x (2,6) = (6,6)
        attn_weigths = torch.softmax(attn_scores/self.div, dim = 1) # (6,6)
        context_vec = attn_weigths @ value #(6,2)
        return context_vec, attn_weigths

In [97]:
sa_ver1 = SelfAttention(3,2)

In [98]:
inputs

tensor([[-0.2728,  0.8483, -0.6835],
        [-0.5035, -0.1803, -1.4679],
        [ 0.1080,  1.9712, -1.5753],
        [-0.5000,  0.6529, -0.7750],
        [-0.2441,  0.4189,  1.3836],
        [-0.3643, -1.2531, -0.5631]])

In [99]:
CV, AW= sa_ver1(inputs)
print(CV)
print(AW)
print(AW.sum())

tensor([[-0.3212, -0.2255],
        [-0.6002, -1.0557],
        [ 0.0438,  0.4991],
        [-0.4321, -0.5035],
        [-0.2450, -0.0611],
        [-0.6531, -1.3012]], grad_fn=<MmBackward0>)
tensor([[0.1731, 0.1311, 0.1832, 0.1638, 0.2199, 0.1289],
        [0.1191, 0.2927, 0.1074, 0.1413, 0.0496, 0.2899],
        [0.1493, 0.0436, 0.1805, 0.1176, 0.4665, 0.0425],
        [0.1649, 0.1778, 0.1672, 0.1669, 0.1488, 0.1744],
        [0.1730, 0.1073, 0.1857, 0.1577, 0.2699, 0.1065],
        [0.0865, 0.3515, 0.0715, 0.1132, 0.0229, 0.3544]],
       grad_fn=<SoftmaxBackward0>)
tensor(6., grad_fn=<SumBackward0>)


In [90]:
import torch
import torch.nn as nn

class SelfAttentionVer2(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()

        self.W_query = nn.Linear(d_in, d_out, bias=False)
        self.W_key   = nn.Linear(d_in, d_out, bias=False)
        self.W_value = nn.Linear(d_in, d_out, bias=False)

        self.scale = d_out ** 0.5

    def forward(self, x):

        query = self.W_query(x)
        key   = self.W_key(x)
        value = self.W_value(x)

        attn_scores = query @ key.T
        attn_weights = torch.softmax(attn_scores / self.scale, dim=1)

        context_vec = attn_weights @ value

        return context_vec, attn_weights

In [91]:
sa_ver2 = SelfAttentionVer2(3,2)
CV, AW= sa_ver2(inputs)
print(CV)
print(AW)
print(AW.sum())

tensor([[-0.0707,  0.3964],
        [-0.0383,  0.3535],
        [ 0.0561,  0.3301],
        [-0.0459,  0.3794],
        [-0.2668,  0.5353],
        [-0.2079,  0.4405]], grad_fn=<MmBackward0>)
tensor([[0.1616, 0.1279, 0.1375, 0.1555, 0.2629, 0.1545],
        [0.1554, 0.1232, 0.1208, 0.1495, 0.2871, 0.1641],
        [0.1516, 0.1012, 0.1147, 0.1418, 0.3506, 0.1401],
        [0.1594, 0.1227, 0.1312, 0.1527, 0.2801, 0.1539],
        [0.1696, 0.1789, 0.1941, 0.1712, 0.1321, 0.1541],
        [0.1627, 0.1676, 0.1519, 0.1635, 0.1739, 0.1803]],
       grad_fn=<SoftmaxBackward0>)
tensor(6., grad_fn=<SumBackward0>)
